In [1]:
from transformers import T5Tokenizer, T5TokenizerFast, T5ForConditionalGeneration, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.metrics import accuracy_score
import torch
from datasets import Dataset
import evaluate
import pandas as pd
import gc
import numpy as np
import json
import nltk
from math import ceil
from utils.config import CV_DATA

In [2]:
new_dataset_path= 'Model_dataset/real_world_data.csv'
old_dataset_path= 'Model_dataset/synthetic_question_ans_data-v2.csv'

#qa model
qa_type_model_name= 't5-small'
qa_type_model_result= '.temp/model_results/fine_tuned_question_answer_model-small_v2'
qa_type_model= '.temp/model/fine_tuned_question_answer_model-small_v2'



In [3]:
CV_DATA.keys()

dict_keys(['availability', 'current_ctc', 'education', 'expected_ctc', 'others', 'personal_information', 'skills', 'working_experience'])

### Preprocessing

In [4]:
data_ratio= {
    "old":20,
    "new":80
    }

new_data_df= pd.read_csv(new_dataset_path)
new_data_df= new_data_df[["question", "question_type", "answer"]]
new_data_df["answer"]= new_data_df["answer"].fillna("")
new_data_df.info()

old_data_df= pd.read_csv(old_dataset_path)
old_data_df= old_data_df[["question", "question_type", "answer"]]
old_data_df["answer"]= old_data_df["answer"].fillna("")
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1882 entries, 0 to 1881
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1882 non-null   object
 1   question_type  1882 non-null   object
 2   answer         1882 non-null   object
dtypes: object(3)
memory usage: 44.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1644 entries, 0 to 1643
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1644 non-null   object
 1   question_type  1644 non-null   object
 2   answer         1644 non-null   object
dtypes: object(3)
memory usage: 38.7+ KB


In [5]:
columns_in_new_df= new_data_df["question_type"].unique()
print(f"columns_in_new_df :{columns_in_new_df}")

columns_in_old_df= old_data_df["question_type"].unique()
print(f"columns_in_old_df :{columns_in_old_df}")

columns_in_new_df :['availability' 'personal_information' 'current_ctc' 'education'
 'working_experience' 'expected_ctc' 'others' 'skills']
columns_in_old_df :['current_ctc' 'expected_ctc' 'personal_information' 'education'
 'working_experience' 'skills' 'availability' 'others']


In [6]:
new_data_df.drop_duplicates(inplace= True)
new_data_df.info()

old_data_df.drop_duplicates(inplace= True)
old_data_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1868 entries, 0 to 1881
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1868 non-null   object
 1   question_type  1868 non-null   object
 2   answer         1868 non-null   object
dtypes: object(3)
memory usage: 58.4+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 1637 entries, 0 to 1643
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       1637 non-null   object
 1   question_type  1637 non-null   object
 2   answer         1637 non-null   object
dtypes: object(3)
memory usage: 51.2+ KB


#### combaing new and old data

In [7]:
new_data_len= len(new_data_df)
total_len= ceil(new_data_len/(data_ratio["new"]/100))
old_data_len= ceil(total_len- new_data_len)
print(total_len)
old_data_len

2335


467

In [8]:
temp_df= pd.DataFrame()
while True:
    temp_df= old_data_df.sample(old_data_len)
    columns_in_old_df= temp_df["question_type"].unique()

    if set(columns_in_old_df)== set(columns_in_new_df):
        break
temp_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 467 entries, 317 to 183
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       467 non-null    object
 1   question_type  467 non-null    object
 2   answer         467 non-null    object
dtypes: object(3)
memory usage: 14.6+ KB


In [9]:
df= pd.concat([new_data_df, temp_df], ignore_index=True)
df.drop_duplicates(inplace= True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2334 entries, 0 to 2334
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       2334 non-null   object
 1   question_type  2334 non-null   object
 2   answer         2334 non-null   object
dtypes: object(3)
memory usage: 72.9+ KB


### mapping content

In [10]:
df["context"] = df["question_type"].map(CV_DATA)

In [11]:
df= df.sample(frac=1).reset_index(drop=True)
df.head()

,question,question_type,answer,context
0,How many years of work experience do you have ...,skills,2 years,Programming Skills\nPython\n Rating out of ...
1,Will you now or in the future require Precisio...,availability,No,Availability for Interviews\nI am available fo...
2,How many years of experience do you have with ...,skills,0 years,Programming Skills\nPython\n Rating out of ...
3,Are you willing to relocate?,availability,Yes,Availability for Interviews\nI am available fo...
4,How many years of experience do you have in wo...,skills,0 years,Programming Skills\nPython\n Rating out of ...


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2334 entries, 0 to 2333
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   question       2334 non-null   object
 1   question_type  2334 non-null   object
 2   answer         2334 non-null   object
 3   context        2334 non-null   object
dtypes: object(4)
memory usage: 73.1+ KB


### Retraing Preparations:

In [13]:
# Preprocess data
def preprocess_data(row):
    input_text = f"question: {row['question']} context: {row['context']}"
    target_text = row['answer']
    return {"input_text": input_text, "target_text": target_text}

processed_data = df.apply(preprocess_data, axis=1)
dataset = Dataset.from_pandas(pd.DataFrame(processed_data.tolist()))

In [14]:
# Split data into train and test
train_test_split = dataset.train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
test_dataset = train_test_split["test"]

In [15]:
tokenizer = T5Tokenizer.from_pretrained(qa_type_model_name)

def tokenize_data(example):
    input_encodings = tokenizer(example["input_text"], truncation=True, padding="max_length", max_length=512)
    target_encodings = tokenizer(example["target_text"], truncation=True, padding="max_length", max_length=128)
    input_encodings["labels"] = target_encodings["input_ids"]
    return input_encodings

train_dataset = train_dataset.map(tokenize_data, batched=True)
test_dataset = test_dataset.map(tokenize_data, batched=True)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

In [16]:
nltk.download('punkt')

# Load metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

def combine_compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them directly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Preprocess for BLEU (expects list of tokens)
    bleu_preds = [pred.split() for pred in decoded_preds]
    bleu_labels = [[label.split()] for label in decoded_labels]  # BLEU expects a list of references

    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rouge_results = {key: value.mid.fmeasure * 100 for key, value in rouge_results.items()}
    
    # Compute BLEU
    bleu_result = bleu_metric.compute(predictions=bleu_preds, references=bleu_labels)
    bleu_score = bleu_result["bleu"] * 100  # Convert BLEU to percentage for consistency

    # Combine results
    combined_results = {
        **rouge_results,
        "bleu": bleu_score
    }
    return combined_results


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # Replace -100 in the labels as we can't decode them directly
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Compute ROUGE
    rouge_results = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    rouge_results = {key: value.mid.fmeasure * 100 for key, value in rouge_results.items()}

    return rouge_results

[nltk_data] Downloading package punkt to /home/manab/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Train

In [17]:
model = T5ForConditionalGeneration.from_pretrained(qa_type_model_name)

In [18]:
training_args = TrainingArguments(
    output_dir=qa_type_model_result,           # Output directory
    eval_strategy="epoch",                     # Evaluate every epoch
    save_strategy="epoch",                     # Save every epoch
    learning_rate=1e-5,                        # Learning rate
    num_train_epochs=50,                       # Number of training epochs
    per_device_train_batch_size=4,             # Batch size during training
    per_device_eval_batch_size=4,              # Batch size during evaluation
    gradient_accumulation_steps=2,             # Gradient accumulation steps
    logging_dir="./logs",                      # Directory for logs
    logging_steps=10,                          # Log every 10 steps
    save_total_limit=4,                        # Limit the number of saved checkpoints
    warmup_steps=800,                          # Warmup steps for learning rate
    weight_decay=0.01,                         # Weight decay for regularization
    adam_epsilon=1e-8,                        # Epsilon for the Adam optimizer
    max_grad_norm=1.0,                        # Max gradient norm for gradient clipping
    # fp16=True,                               # Enable mixed precision training (optional)
    # use_cpu=True,                            # Force CPU usage (if necessary)
    load_best_model_at_end=True,               # Load best model at the end of training
    # metric_for_best_model="rougeL",         # Metric to monitor for best model
    # greater_is_better=True,                    # Set to True for accuracy metrics
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    # compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)


# Clearing memory before training
torch.cuda.empty_cache()
gc.collect()

# Train the model
trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,11.693900,14.319910
2,0.540300,0.211042
3,0.162300,0.120702
4,0.060800,0.043983
5,0.039000,0.037106
6,0.036900,0.032755
7,0.039100,0.030249
8,0.034300,0.029200
9,0.037100,0.028198
10,0.028000,0.026915


In [ ]:
evaluation_results = trainer.evaluate()
evaluation_results

In [ ]:
trainer.save_model(qa_type_model)
tokenizer.save_pretrained(qa_type_model)